# 01C — Lead Scoring SQL Optimizer
Optimiza la query que construye el universo live antes del scoring. Read-only por defecto.


In [58]:
from __future__ import annotations
import sys, time
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Ejecuta este notebook dentro de bd_replica_crm.")

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from replica_cygnus.settings import load_settings
from replica_cygnus.connections import connect_postgres

settings = load_settings(PROJECT_ROOT)
conn = connect_postgres(settings)

pd.set_option("display.max_columns",150)
pd.set_option("display.max_rows",150)
pd.set_option("display.width",220)

def df(sql, params=None):
    return pd.read_sql_query(sql, conn, params=params)

print("DB:", settings.postgres.database)

from replica_cygnus.lead_scoring.config import load_lead_scoring_config
from replica_cygnus.lead_scoring.training import MODEL_FEATURES

config_path=PROJECT_ROOT/"config"/"lead_scoring.yml"
if not config_path.exists():
    config_path=PROJECT_ROOT/"config"/"lead_scoring.example.yml"
cfg=load_lead_scoring_config(config_path)
print("score_window_days:",cfg.score_window_days)


DB: medallio_dw
score_window_days: 14


## 1. Query productiva actual


In [59]:
feature_cols=", ".join([f for f in MODEL_FEATURES if f not in {"codigo_proyecto","asesor","canal","medio"}])
base_select=f"""
SELECT evidence_key,lead_id,decision_at,documento_cliente,codigo_proyecto,
       asesor,canal,medio,{feature_cols}
FROM features.lead_evidence e
"""
current_where=f"""
WHERE decision_at >= current_date - ({int(cfg.score_window_days)} * interval '1 day')
  AND features_refreshed_at IS NOT NULL
  AND NOT EXISTS (
      SELECT 1
      FROM core.fact_ciclo_comercial_unidad c
      WHERE c.documento_cliente=e.documento_cliente
        AND COALESCE(c.codigo_proyecto_ciclo,c.codigo_proyecto_unidad)=e.codigo_proyecto
        AND (
          c.fecha_separacion BETWEEN e.decision_at::date AND current_date
          OR c.fecha_venta BETWEEN e.decision_at::date AND current_date
        )
  )
"""
current_query=base_select+current_where+" ORDER BY decision_at,evidence_key"
print(current_query)



SELECT evidence_key,lead_id,decision_at,documento_cliente,codigo_proyecto,
       asesor,canal,medio,hour_of_day, day_of_week, is_weekend, client_prior_assignments_90d, days_since_previous_assignment, project_leads_90d, project_sep_rate_90d, project_minuta_rate_180d, advisor_leads_90d, advisor_sep_rate_90d, advisor_minuta_rate_180d, global_sep_rate_90d, global_minuta_rate_180d
FROM features.lead_evidence e

WHERE decision_at >= current_date - (14 * interval '1 day')
  AND features_refreshed_at IS NOT NULL
  AND NOT EXISTS (
      SELECT 1
      FROM core.fact_ciclo_comercial_unidad c
      WHERE c.documento_cliente=e.documento_cliente
        AND COALESCE(c.codigo_proyecto_ciclo,c.codigo_proyecto_unidad)=e.codigo_proyecto
        AND (
          c.fecha_separacion BETWEEN e.decision_at::date AND current_date
          OR c.fecha_venta BETWEEN e.decision_at::date AND current_date
        )
  )
 ORDER BY decision_at,evidence_key


## 2. Selectividad por etapa


In [60]:
counts={}
counts["recent"]=int(df(f"""
SELECT COUNT(*) FROM features.lead_evidence
WHERE decision_at >= current_date - ({int(cfg.score_window_days)} * interval '1 day')
""").iloc[0,0])
counts["recent_with_features"]=int(df(f"""
SELECT COUNT(*) FROM features.lead_evidence
WHERE decision_at >= current_date - ({int(cfg.score_window_days)} * interval '1 day')
  AND features_refreshed_at IS NOT NULL
""").iloc[0,0])
counts["core_rows"]=int(df("SELECT COUNT(*) FROM core.fact_ciclo_comercial_unidad").iloc[0,0])
pd.DataFrame(counts.items(),columns=["stage","rows"])


C:\Users\dinat\AppData\Local\Temp\ipykernel_32968\126482159.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn, params=params)


,stage,rows
0,recent,1930
1,recent_with_features,0
2,core_rows,2009


## 3. EXPLAIN actual


In [61]:
plan=df("EXPLAIN "+current_query)
print("\n".join(plan.iloc[:,0].astype(str)))


Incremental Sort  (cost=6832.93..6832.98 rows=2 width=212)
  Sort Key: e.decision_at, e.evidence_key
  Presorted Key: e.decision_at
  ->  Nested Loop Anti Join  (cost=1052.14..6832.92 rows=1 width=212)
        Join Filter: ((c.documento_cliente = e.documento_cliente) AND (COALESCE(c.codigo_proyecto, u.codigo_proyecto) = e.codigo_proyecto) AND (((c.fecha_separacion >= (e.decision_at)::date) AND (c.fecha_separacion <= CURRENT_DATE)) OR ((c.fecha_venta >= (e.decision_at)::date) AND (c.fecha_venta <= CURRENT_DATE))))
        ->  Index Scan Backward using ix_lead_evidence_decision_at on lead_evidence e  (cost=0.42..5588.79 rows=1 width=212)
              Index Cond: (decision_at >= (CURRENT_DATE - '14 days'::interval))
              Filter: (features_refreshed_at IS NOT NULL)
        ->  Hash Left Join  (cost=1051.71..1153.72 rows=2009 width=990)
              Hash Cond: (c.codigo_proforma = m.codigo_proforma)
              ->  Hash Left Join  (cost=171.83..266.30 rows=2009 width=34)
      

C:\Users\dinat\AppData\Local\Temp\ipykernel_32968\126482159.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn, params=params)


## 4. Índices disponibles


In [62]:
indexes=df("""
SELECT schemaname,tablename,indexname,indexdef
FROM pg_indexes
WHERE (schemaname='features' AND tablename='lead_evidence')
   OR (schemaname='core' AND tablename='fact_ciclo_comercial_unidad')
ORDER BY schemaname,tablename,indexname
""")
indexes


C:\Users\dinat\AppData\Local\Temp\ipykernel_32968\126482159.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn, params=params)


,schemaname,tablename,indexname,indexdef
0,features,lead_evidence,ix_lead_evidence_advisor_time,CREATE INDEX ix_lead_evidence_advisor_time ON ...
1,features,lead_evidence,ix_lead_evidence_decision_at,CREATE INDEX ix_lead_evidence_decision_at ON f...
2,features,lead_evidence,ix_lead_evidence_document_time,CREATE INDEX ix_lead_evidence_document_time ON...
3,features,lead_evidence,ix_lead_evidence_project_time,CREATE INDEX ix_lead_evidence_project_time ON ...
4,features,lead_evidence,lead_evidence_pkey,CREATE UNIQUE INDEX lead_evidence_pkey ON feat...


## 5. Perfil de core


In [63]:
core_profile=df("""
SELECT COUNT(*) rows,
       COUNT(*) FILTER (WHERE documento_cliente IS NULL) null_documento,
       COUNT(*) FILTER (WHERE codigo_proyecto_ciclo IS NULL) null_codigo_ciclo,
       COUNT(*) FILTER (WHERE codigo_proyecto_unidad IS NULL) null_codigo_unidad,
       COUNT(*) FILTER (WHERE fecha_separacion IS NOT NULL) with_sep,
       COUNT(*) FILTER (WHERE fecha_venta IS NOT NULL) with_venta
FROM core.fact_ciclo_comercial_unidad
""")
core_profile.T


C:\Users\dinat\AppData\Local\Temp\ipykernel_32968\126482159.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn, params=params)


,0
rows,2009
null_documento,0
null_codigo_ciclo,0
null_codigo_unidad,0
with_sep,2009
with_venta,1362


## 6. Variante sin COALESCE


In [64]:
alt_where_or=f"""
WHERE decision_at >= current_date - ({int(cfg.score_window_days)} * interval '1 day')
  AND features_refreshed_at IS NOT NULL
  AND NOT EXISTS (
      SELECT 1
      FROM core.fact_ciclo_comercial_unidad c
      WHERE c.documento_cliente=e.documento_cliente
        AND (
             c.codigo_proyecto_ciclo=e.codigo_proyecto
          OR (c.codigo_proyecto_ciclo IS NULL AND c.codigo_proyecto_unidad=e.codigo_proyecto)
        )
        AND (
          c.fecha_separacion BETWEEN e.decision_at::date AND current_date
          OR c.fecha_venta BETWEEN e.decision_at::date AND current_date
        )
  )
"""
alt_query_or=base_select+alt_where_or+" ORDER BY decision_at,evidence_key"
plan_alt=df("EXPLAIN "+alt_query_or)
print("\n".join(plan_alt.iloc[:,0].astype(str)))


C:\Users\dinat\AppData\Local\Temp\ipykernel_32968\126482159.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn, params=params)


Sort  (cost=6770.20..6770.20 rows=1 width=212)
  Sort Key: e.decision_at, e.evidence_key
  ->  Hash Right Anti Join  (cost=6640.52..6770.19 rows=1 width=212)
        Hash Cond: (c.documento_cliente = e.documento_cliente)
        Join Filter: (((c.codigo_proyecto = e.codigo_proyecto) OR ((c.codigo_proyecto IS NULL) AND (u.codigo_proyecto = e.codigo_proyecto))) AND (((c.fecha_separacion >= (e.decision_at)::date) AND (c.fecha_separacion <= CURRENT_DATE)) OR ((c.fecha_venta >= (e.decision_at)::date) AND (c.fecha_venta <= CURRENT_DATE))))
        ->  Hash Left Join  (cost=1051.71..1153.72 rows=2009 width=990)
              Hash Cond: (c.codigo_proforma = m.codigo_proforma)
              ->  Hash Left Join  (cost=171.83..266.30 rows=2009 width=34)
                    Hash Cond: (c.codigo_unidad = u.codigo_unidad)
                    ->  Seq Scan on int_ciclo_comercial_unidad c  (cost=0.00..89.18 rows=2009 width=39)
                          Filter: ((fecha_separacion <= CURRENT_DATE) OR (fec

## 7. Variante prefiltrando eventos


In [65]:
alt_query_prefilter=f"""
WITH core_events AS (
    SELECT documento_cliente,codigo_proyecto_ciclo,codigo_proyecto_unidad,fecha_separacion,fecha_venta
    FROM core.fact_ciclo_comercial_unidad
    WHERE fecha_separacion IS NOT NULL OR fecha_venta IS NOT NULL
)
{base_select}
WHERE decision_at >= current_date - ({int(cfg.score_window_days)} * interval '1 day')
  AND features_refreshed_at IS NOT NULL
  AND NOT EXISTS (
      SELECT 1 FROM core_events c
      WHERE c.documento_cliente=e.documento_cliente
        AND (
             c.codigo_proyecto_ciclo=e.codigo_proyecto
          OR (c.codigo_proyecto_ciclo IS NULL AND c.codigo_proyecto_unidad=e.codigo_proyecto)
        )
        AND (
          c.fecha_separacion BETWEEN e.decision_at::date AND current_date
          OR c.fecha_venta BETWEEN e.decision_at::date AND current_date
        )
  )
ORDER BY decision_at,evidence_key
"""
plan_pref=df("EXPLAIN "+alt_query_prefilter)
print("\n".join(plan_pref.iloc[:,0].astype(str)))


Sort  (cost=6770.20..6770.20 rows=1 width=212)
  Sort Key: e.decision_at, e.evidence_key
  ->  Hash Right Anti Join  (cost=6640.52..6770.19 rows=1 width=212)
        Hash Cond: (c.documento_cliente = e.documento_cliente)
        Join Filter: (((c.codigo_proyecto = e.codigo_proyecto) OR ((c.codigo_proyecto IS NULL) AND (u.codigo_proyecto = e.codigo_proyecto))) AND (((c.fecha_separacion >= (e.decision_at)::date) AND (c.fecha_separacion <= CURRENT_DATE)) OR ((c.fecha_venta >= (e.decision_at)::date) AND (c.fecha_venta <= CURRENT_DATE))))
        ->  Hash Left Join  (cost=1051.71..1153.72 rows=2009 width=990)
              Hash Cond: (c.codigo_proforma = m.codigo_proforma)
              ->  Hash Left Join  (cost=171.83..266.30 rows=2009 width=34)
                    Hash Cond: (c.codigo_unidad = u.codigo_unidad)
                    ->  Seq Scan on int_ciclo_comercial_unidad c  (cost=0.00..89.18 rows=2009 width=39)
                          Filter: (((fecha_separacion IS NOT NULL) OR (fecha_

C:\Users\dinat\AppData\Local\Temp\ipykernel_32968\126482159.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn, params=params)


## 8. Benchmark controlado


In [66]:
BENCH_LIMIT=5000
bench=[]
for name,q in [("current",current_query),("alt_or",alt_query_or),("prefilter",alt_query_prefilter)]:
    limited=f"SELECT * FROM ({q}) q LIMIT {BENCH_LIMIT}"
    t0=time.perf_counter()
    try:
        out=df(limited); s=time.perf_counter()-t0
        bench.append({"query":name,"seconds":s,"rows":len(out),"rows_per_second":len(out)/s if s else np.nan})
    except Exception as exc:
        conn.rollback(); bench.append({"query":name,"seconds":np.nan,"rows":0,"error":repr(exc)})
benchmark_table=pd.DataFrame(bench).sort_values("seconds")
benchmark_table


C:\Users\dinat\AppData\Local\Temp\ipykernel_32968\126482159.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn, params=params)
C:\Users\dinat\AppData\Local\Temp\ipykernel_32968\126482159.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn, params=params)
C:\Users\dinat\AppData\Local\Temp\ipykernel_32968\126482159.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn, params=params)


,query,seconds,rows,rows_per_second
1,alt_or,0.012033,0,0.0
2,prefilter,0.012554,0,0.0
0,current,0.015796,0,0.0


In [67]:
diagnostico_universo = df(f"""
WITH base AS (
    SELECT
        e.*,
        EXISTS (
            SELECT 1
            FROM core.fact_ciclo_comercial_unidad c
            WHERE c.documento_cliente = e.documento_cliente
              AND COALESCE(
                    c.codigo_proyecto_ciclo,
                    c.codigo_proyecto_unidad
                  ) = e.codigo_proyecto
              AND (
                    c.fecha_separacion
                        BETWEEN e.decision_at::date AND current_date
                 OR c.fecha_venta
                        BETWEEN e.decision_at::date AND current_date
              )
        ) AS excluido_por_conversion
    FROM features.lead_evidence e
    WHERE e.decision_at >=
          current_date - ({int(cfg.score_window_days)} * interval '1 day')
)
SELECT
    COUNT(*) AS recientes,
    COUNT(*) FILTER (
        WHERE features_refreshed_at IS NOT NULL
    ) AS con_features,
    COUNT(*) FILTER (
        WHERE features_refreshed_at IS NULL
    ) AS sin_features,
    COUNT(*) FILTER (
        WHERE documento_cliente IS NULL
    ) AS sin_documento,
    COUNT(*) FILTER (
        WHERE codigo_proyecto IS NULL
    ) AS sin_proyecto,
    COUNT(*) FILTER (
        WHERE excluido_por_conversion
    ) AS excluidos_conversion,
    COUNT(*) FILTER (
        WHERE features_refreshed_at IS NOT NULL
          AND NOT excluido_por_conversion
    ) AS scoring_candidates
FROM base
""")

diagnostico_universo.T

C:\Users\dinat\AppData\Local\Temp\ipykernel_32968\126482159.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn, params=params)


,0
recientes,1930
con_features,0
sin_features,1930
sin_documento,0
sin_proyecto,0
excluidos_conversion,7
scoring_candidates,0


In [68]:
df("""
SELECT
    evidence_source,
    COUNT(*) AS n,
    MIN(decision_at) AS min_decision_at,
    MAX(decision_at) AS max_decision_at,
    MIN(features_refreshed_at) AS min_refreshed,
    MAX(features_refreshed_at) AS max_refreshed
FROM features.lead_evidence
GROUP BY evidence_source
ORDER BY n DESC
""")

C:\Users\dinat\AppData\Local\Temp\ipykernel_32968\126482159.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn, params=params)


,evidence_source,n,min_decision_at,max_decision_at,min_refreshed,max_refreshed
0,BACKFILL_INFERRED,203856,2019-10-02 23:08:14.794272+00:00,2026-08-23 04:52:47.821190+00:00,None,None
1,LIVE,2091,2026-08-23 05:00:08.868356+00:00,2026-09-07 00:04:08.133367+00:00,None,None


In [69]:
df("""
SELECT
    COUNT(*) AS pendientes,
    MIN(decision_at) AS desde,
    MAX(decision_at) AS hasta
FROM features.lead_evidence
WHERE features_refreshed_at IS NULL
""").T

C:\Users\dinat\AppData\Local\Temp\ipykernel_32968\126482159.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn, params=params)


,0
pendientes,205947
desde,2019-10-02 23:08:14.794272+00:00
hasta,2026-09-07 00:04:08.133367+00:00


In [71]:
df(f"""
SELECT
    COUNT(*) AS pendientes_live
FROM features.lead_evidence
WHERE features_refreshed_at IS NULL
  AND decision_at >=
      current_date - ({int(cfg.score_window_days)} * interval '1 day')
""")

C:\Users\dinat\AppData\Local\Temp\ipykernel_32968\126482159.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn, params=params)


,pendientes_live
0,1930


## 9. EXPLAIN ANALYZE opcional


In [53]:
RUN_EXPLAIN_ANALYZE=False
if RUN_EXPLAIN_ANALYZE:
    for name,q in [("current",current_query),("alt_or",alt_query_or),("prefilter",alt_query_prefilter)]:
        print("\n###",name)
        p=df("EXPLAIN (ANALYZE,BUFFERS) "+q)
        print("\n".join(p.iloc[:,0].astype(str)))
else:
    print("RUN_EXPLAIN_ANALYZE=False")


RUN_EXPLAIN_ANALYZE=False


## 10. Candidatos de índices
No se crean automáticamente. Revisa primero `EXPLAIN` y benchmark.


In [54]:
index_candidates=pd.DataFrame([
{"table":"core.fact_ciclo_comercial_unidad","index":"(documento_cliente, codigo_proyecto_ciclo, fecha_separacion)","reason":"anti-join separación"},
{"table":"core.fact_ciclo_comercial_unidad","index":"(documento_cliente, codigo_proyecto_unidad, fecha_separacion)","reason":"fallback separación"},
{"table":"core.fact_ciclo_comercial_unidad","index":"(documento_cliente, codigo_proyecto_ciclo, fecha_venta)","reason":"anti-join venta"},
{"table":"core.fact_ciclo_comercial_unidad","index":"(documento_cliente, codigo_proyecto_unidad, fecha_venta)","reason":"fallback venta"},
{"table":"features.lead_evidence","index":"(decision_at DESC)","reason":"ventana live"},
])
index_candidates


,table,index,reason
0,core.fact_ciclo_comercial_unidad,"(documento_cliente, codigo_proyecto_ciclo, fec...",anti-join separación
1,core.fact_ciclo_comercial_unidad,"(documento_cliente, codigo_proyecto_unidad, fe...",fallback separación
2,core.fact_ciclo_comercial_unidad,"(documento_cliente, codigo_proyecto_ciclo, fec...",anti-join venta
3,core.fact_ciclo_comercial_unidad,"(documento_cliente, codigo_proyecto_unidad, fe...",fallback venta
4,features.lead_evidence,(decision_at DESC),ventana live


## 11. Diagnóstico automático


In [55]:
diag=[]
def add(sev,area,msg): diag.append({"severity":sev,"area":area,"message":msg})

b=benchmark_table.dropna(subset=["seconds"])
if len(b):
    best=b.iloc[0]
    cur=b[b["query"].eq("current")]
    if len(cur):
        cs=float(cur.iloc[0]["seconds"]); bs=float(best["seconds"])
        if best["query"]!="current" and bs<cs*.8:
            add("HIGH","sql_rewrite",f"{best['query']} mejora {cs:.2f}s → {bs:.2f}s")
        else:
            add("INFO","sql_rewrite","No hay mejora fuerte con variantes simples")
if counts["core_rows"]>1_000_000:
    add("HIGH","core_volume",f"core tiene {counts['core_rows']:,} filas")
if counts["recent_with_features"]>10000:
    add("INFO","live_volume",f"Universo live con features: {counts['recent_with_features']:,}")
diagnostic=pd.DataFrame(diag)
diagnostic


,severity,area,message
0,INFO,sql_rewrite,No hay mejora fuerte con variantes simples


In [56]:
conn.close(); print("Conexión cerrada.")


Conexión cerrada.
